# Financial News Semantic Search Engine

A semantic search tool over financial news: sentence-transformer embeddings, cosine-similarity retrieval, and a Gradio interface returning the most relevant articles for a natural-language query.

---

### Key Insights 

**Overview**

This project applies sentence embeddings  to a real-world
financial news dataset. It involves two steps: (1) cleaning the
text by extracting URLs into a separate column, and (2) building
a Gradio-powered semantic search tool that returns the top 5
most similar news headlines to any user query.

**Key Insight 1 – Semantic search vs keyword search**

A keyword search for 'earnings surprise' would only return
headlines containing those exact words. Semantic search using
cosine similarity on sentence embeddings returns headlines that
mean the same thing — even if they use completely different
words like 'quarterly beat' or 'profit outperformance'.

**Key Insight 2 – Why cosine similarity and not Euclidean distance**

Embeddings are high-dimensional vectors. Cosine similarity
measures the angle between two vectors, not their distance.
This makes it robust to the length of the text — a short
headline and a long one about the same topic still get a
high similarity score.

**Key Insight 3 – Pre-compute embeddings once**

Encoding 16,990 headlines takes time. We encode the entire
dataset once before launching Gradio, storing the result.
Each user query then only needs to encode one sentence and
compare against the stored embeddings — making the search
near-instant.

**Personal Takeaway**

This project showed how sentence embeddings — introduced in L3
as a mathematical concept — translate directly into a practical
business tool. A financial analyst could use this to instantly
find all news about a specific event type across thousands of
headlines, without manually reading each one.


## Step 0 – Install required libraries

**After running this cell, select Runtime → Restart session before continuing.**


In [2]:
# Install required libraries — same as L3 notebook plus gradio for the UI
!pip install sentence-transformers scikit-learn gradio -q

## Step 1 – Upload and Load the Dataset

Run this cell. A **'Choose Files'** button will appear.
Click it and select your CSV file (e.g. `financial_news.csv`).
The code will automatically detect the filename.

**This works with any CSV that has a `text` column.**


In [3]:
import pandas as pd
import io
from google.colab import files

# Open a file-picker dialog
uploaded = files.upload()

# Read whichever file the user chose — works for any filename
filename = next(iter(uploaded))
print(f'File uploaded: {filename}')

df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f'Shape       : {df.shape}')
print(f'Columns     : {df.columns.tolist()}')
df.head()

Saving financial_news.csv to financial_news.csv
File uploaded: financial_news.csv
Shape       : (16990, 2)
Columns     : ['text', 'label']


,text,label
0,Here are Thursday's biggest analyst calls: App...,0
1,Buy Las Vegas Sands as travel to Singapore bui...,0
2,"Piper Sandler downgrades DocuSign to sell, cit...",0
3,"Analysts react to Tesla's latest earnings, bre...",0
4,Netflix and its peers are set for a ‘return to...,0


## Step 2 – Extract URLs from the `text` column

The task states: *"The last part of the sentence in each row of `text`
contains a URL. Remove this from `text` and create a new column called
`URL` and add the URL."*

We use a regular expression to find any URL pattern (`http://` or
`https://` followed by non-space characters) anywhere in the text.
The URL is extracted into a new column, then removed from the original
text. Rows that contain no URL are left unchanged — the `URL` column
for those rows will be empty.


In [4]:
import re

# Regular expression pattern to match any URL (http or https)
URL_PATTERN = r'https?://\S+'

def extract_url(text):
    """Finds and returns the URL from a text string. Returns '' if no URL found."""
    match = re.search(URL_PATTERN, str(text))
    return match.group(0) if match else ''

def remove_url(text):
    """Removes the URL from a text string and strips trailing whitespace."""
    return re.sub(URL_PATTERN, '', str(text)).strip()

# Create the new URL column
df['URL'] = df['text'].apply(extract_url)

# Overwrite text with the cleaned version (URL removed)
df['text'] = df['text'].apply(remove_url)

# Verify the result
print(f'Rows with a URL extracted : {(df["URL"] != "").sum()}')
print(f'Rows with no URL          : {(df["URL"] == "").sum()}')
print()
df[['text', 'URL']].head(10)

Rows with a URL extracted : 15067
Rows with no URL          : 1923



,text,URL
0,Here are Thursday's biggest analyst calls: App...,https://t.co/QPN8Gwl7Uh
1,Buy Las Vegas Sands as travel to Singapore bui...,https://t.co/fLS2w57iCz
2,"Piper Sandler downgrades DocuSign to sell, cit...",https://t.co/1EmtywmYpr
3,"Analysts react to Tesla's latest earnings, bre...",https://t.co/kwhoE6W06u
4,Netflix and its peers are set for a ‘return to...,https://t.co/jPpdl0D9s4
5,Barclays believes earnings for these underperf...,https://t.co/PHbsyVGAyE
6,"Bernstein upgrades Alibaba, says shares can ra...",https://t.co/m3ApoPRGU0
7,"Analysts react to Netflix's strong quarter, wi...",https://t.co/cQngJsyefD
8,Buy Chevron as shares look attractive at these...,https://t.co/GkDpFvxjEP
9,Morgan Stanley says these global stocks are se...,https://t.co/GeWxa5YoWr


## Step 3 – Create Sentence Embeddings

Following exactly the approach taught in **L3_Embeddings_Words_to_Sentences.ipynb**:

1. Load the pre-trained `all-MiniLM-L6-v2` model from `sentence_transformers`
2. Call `sentence_model.encode()` on the cleaned `text` column

This encodes all headlines into 384-dimensional vectors that capture
semantic meaning. We pre-compute all embeddings once here — so the
Gradio search tool can respond instantly to any query.

**This may take 1–2 minutes for ~17,000 headlines.**


In [5]:
# Import libraries — exactly as shown in L3
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load the same pre-trained sentence-transformer model used in L3
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all cleaned headlines
# show_progress_bar=True displays a progress bar during encoding
print('Encoding headlines — this may take 1–2 minutes...')
corpus_embeddings = sentence_model.encode(
    df['text'].tolist(),
    show_progress_bar=True
)

print(f'\nEmbeddings shape: {corpus_embeddings.shape}')
print(f'Each headline is now a vector of {corpus_embeddings.shape[1]} dimensions.')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding headlines — this may take 1–2 minutes...


Batches:   0%|          | 0/531 [00:00<?, ?it/s]


Embeddings shape: (16990, 384)
Each headline is now a vector of 384 dimensions.


## Step 4 – Verify Cosine Similarity

Before building the Gradio tool, we verify that cosine similarity
works correctly on our embeddings — exactly as demonstrated in L3
where similarity between two sentences was computed and printed.


In [6]:
# Compute similarity between the first two headlines — same pattern as L3
similarity = cosine_similarity(
    [corpus_embeddings[0]],
    [corpus_embeddings[1]]
)[0][0]

print(f'Headline 1: {df["text"].iloc[0]}')
print(f'Headline 2: {df["text"].iloc[1]}')
print(f'Cosine similarity: {similarity:.4f}')

Headline 1: Here are Thursday's biggest analyst calls: Apple, Amazon, Tesla, Palantir, DocuSign, Exxon &amp; more
Headline 2: Buy Las Vegas Sands as travel to Singapore builds, Wells Fargo says
Cosine similarity: 0.0996


## Step 5 – Build the Semantic Search Tool with Gradio

The task asks for a Gradio interface where the user enters a query
(e.g. `"earnings surprise"` or `"regulatory fine"`) and receives
the **top 5 most semantically similar** headlines as output.

The search function:
1. Encodes the user's query into a 384-dimensional vector
2. Computes cosine similarity between the query vector and all
   pre-computed headline embeddings
3. Sorts by similarity (highest first) and returns the top 5
4. Displays the headline text, its URL, and the similarity score


In [7]:
import gradio as gr

def semantic_search(query):
    """
    Takes a user query string, encodes it using the same sentence
    transformer model, computes cosine similarity against all
    pre-computed headline embeddings, and returns the top 5 results.
    """
    # Step 1: Encode the user query — same sentence_model.encode() as L3
    query_embedding = sentence_model.encode([query])

    # Step 2: Compute cosine similarity between query and all headlines
    # cosine_similarity returns a (1, N) array — we take the first row
    similarities = cosine_similarity(query_embedding, corpus_embeddings)[0]

    # Step 3: Get the indices of the top 5 highest similarity scores
    # np.argsort sorts ascending — [::-1] reverses to descending
    top5_indices = np.argsort(similarities)[::-1][:5]

    # Step 4: Build the results table
    results = []
    for rank, idx in enumerate(top5_indices, start=1):
        results.append({
            'Rank'       : rank,
            'Similarity' : round(float(similarities[idx]), 4),
            'Headline'   : df['text'].iloc[idx],
            'URL'        : df['URL'].iloc[idx] if df['URL'].iloc[idx] != '' else 'N/A'
        })

    return pd.DataFrame(results)

# Build the Gradio interface
demo = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(
        label='Enter your search query',
        placeholder='e.g. earnings surprise, regulatory fine, Fed interest rate...'
    ),
    outputs=gr.DataFrame(label='Top 5 Most Similar Headlines'),
    title='Financial News Semantic Search',
    description=(
        'Enter any financial topic or phrase. '
        'The tool returns the 5 most semantically similar headlines '
        'from the financial_news dataset, ranked by cosine similarity.'
    ),
    examples=[
        ['earnings surprise'],
        ['regulatory fine'],
        ['interest rate hike'],
        ['stock downgrade'],
        ['merger acquisition']
    ]
)

# Launch the Gradio app
# share=True generates a public link you can open in any browser
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26ffabaa4ad6a711ee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
